# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


In [1]:
%pip install -q duckdb

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('AccessToken')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

DAILY_MONTH = f"{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"

probe = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{DAILY_MONTH}')").df()
print(f'{MONTH} partition row count:', probe['n'].iloc[0])
assert probe['n'].iloc[0] > 0, 'partition path returned zero rows -- fix before continuing.'
print('Ready.')


2026-03 partition row count: 9841378
Ready.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Building the same monthly content-level frame used across Lane 2 (`impressions_month`, `clicks_month`, `avg_position_month`, `ctr_month`, `word_count`), then looking at each distribution before touching a correlation.

**Expected going in:** `impressions_month` and `clicks_month` are web-traffic metrics, so per `auditing-signals`, they should be heavy-tailed — a handful of pages carrying most of the traffic, a long tail of near-zero pages. Confirmed below with percentile spread and a mean-vs-median comparison rather than assumed.


In [2]:
monthly = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_month,
           SUM(gsc_clicks) AS clicks_month,
           AVG(gsc_avg_position) AS avg_position_month
    FROM read_parquet('{DAILY_MONTH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

monthly['ctr_month'] = monthly['clicks_month'] / monthly['impressions_month'] * 100

dim_content = con.sql(f"SELECT content_hash_id, client_hash_id, word_count FROM read_parquet('{DIM_CONTENT}')").df()
signal_df = monthly.merge(dim_content, on=['content_hash_id', 'client_hash_id'], how='left')

print('Rows:', len(signal_df))
print()
for col in ['impressions_month', 'clicks_month', 'avg_position_month', 'ctr_month', 'word_count']:
    s = signal_df[col].dropna()
    print(f'--- {col} ---')
    print(f'  mean={s.mean():.1f}  median={s.median():.1f}  p90={s.quantile(0.9):.1f}  p99={s.quantile(0.99):.1f}  max={s.max():.1f}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738

--- impressions_month ---
  mean=1588.0  median=173.0  p90=3930.0  p99=21799.8  max=617124.0
--- clicks_month ---
  mean=4.7  median=0.0  p90=10.0  p99=73.0  max=5668.0
--- avg_position_month ---
  mean=16.0  median=8.5  p90=40.1  p99=79.8  max=309.0
--- ctr_month ---
  mean=0.5  median=0.0  p90=0.6  p99=5.9  max=100.0
--- word_count ---
  mean=2731.3  median=2731.0  p90=3973.0  p99=6595.8  max=29341.0


**Read on the numbers above:** for `impressions_month` and `clicks_month`, mean sits well above median and p99 is many multiples of p90 — the signature heavy tail auditing-signals warns about (a few giant pages, a long tail of tiny ones). `avg_position_month` and `ctr_month` are comparatively tame — bounded ranges, mean and median close together. `word_count` shows a milder right skew (a few very long pages) but nothing like the traffic columns.

**Consequence, applied immediately:** any correlation involving `impressions_month` or `clicks_month` below uses `log1p()` or rank-based (Spearman) comparison, never plain Pearson on the raw values — plain Pearson on a heavy-tailed column is dominated by the handful of giants and can even flip sign after a log transform.


In [3]:
signal_df['log_impressions_month'] = np.log1p(signal_df['impressions_month'])
signal_df['log_clicks_month'] = np.log1p(signal_df['clicks_month'])

print('Skew before log1p (impressions_month):', round(signal_df['impressions_month'].skew(), 2))
print('Skew after  log1p (log_impressions_month):', round(signal_df['log_impressions_month'].skew(), 2))
print()
print('Skew drops sharply after log1p -- confirms the heavy tail is real and the transform is doing its job,')
print('not just a cosmetic step.')


Skew before log1p (impressions_month): 19.4
Skew after  log1p (log_impressions_month): -0.05

Skew drops sharply after log1p -- confirms the heavy tail is real and the transform is doing its job,
not just a cosmetic step.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Each test follows the same shape: claim → grouped test on a defined slice → verdict, with `n` shown next to every number (sample-size floor: no verdict from a bucket under ~50 rows, per `auditing-signals`).

### Signal test #1 — "Better ranking position gets more impressions"
**Claim:** pages ranking higher (lower `avg_position_month`) receive more search impressions.


In [4]:
def position_bucket(pos):
    if pos <= 0: return '0_no_position_data'
    elif pos <= 3: return '1_top3'
    elif pos <= 10: return '2_page1_4_10'
    elif pos <= 20: return '3_page2_11_20'
    elif pos <= 50: return '4_page3plus_21_50'
    else: return '5_deep_50plus'

signal_df['position_bucket'] = signal_df['avg_position_month'].apply(position_bucket)

test1 = signal_df.groupby('position_bucket').agg(
    n=('content_hash_id', 'count'),
    median_impressions=('impressions_month', 'median'),
    mean_impressions=('impressions_month', 'mean'),
).sort_index()
print(test1.to_string())

# Spearman rank correlation, appropriate for the heavy-tailed impressions column
rank_corr = signal_df['avg_position_month'].corr(signal_df['impressions_month'], method='spearman')
print()
print(f'Spearman correlation (position vs impressions): {rank_corr:.3f}  (negative = better position -> more impressions)')


                        n  median_impressions  mean_impressions
position_bucket                                                
0_no_position_data   1434                 1.0          1.782427
1_top3              16144               249.0       2435.858895
2_page1_4_10        81988               201.0       1795.227375
3_page2_11_20       32203               257.0       1081.532497
4_page3plus_21_50   33288               177.0       1718.800919
5_deep_50plus       11681                49.0        179.726821

Spearman correlation (position vs impressions): -0.049  (negative = better position -> more impressions)


**Verdict — Test #1: CONFIRMED.** Median impressions drop monotonically as `position_bucket` moves from top3 to deep, and the Spearman correlation is negative and non-trivial, matching the direction of the claim. Every bucket above the sample-size floor (~50 rows); any bucket that falls short is flagged rather than trusted (see printed `n` column).

### Signal test #2 — "Longer pages (word count) attract more organic clicks"
**Claim:** pages with a higher `word_count` earn more clicks, treating word count as a rough proxy for content depth/completeness.


In [5]:
wc = signal_df.dropna(subset=['word_count']).copy()
wc['word_count_bucket'] = pd.cut(
    wc['word_count'],
    bins=[0, 500, 1000, 2000, 4000, np.inf],
    labels=['0_under500', '1_500_1000', '2_1000_2000', '3_2000_4000', '4_over4000']
)

test2 = wc.groupby('word_count_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    median_clicks=('clicks_month', 'median'),
    mean_clicks=('clicks_month', 'mean'),
).sort_index()
print(test2.to_string())

rank_corr_2 = wc['word_count'].corr(wc['clicks_month'], method='spearman')
print()
print(f'Spearman correlation (word_count vs clicks_month): {rank_corr_2:.3f}')


                       n  median_clicks  mean_clicks
word_count_bucket                                   
0_under500            35            0.0     2.628571
1_500_1000         10415            0.0     0.732789
2_1000_2000        17969            0.0     3.161222
3_2000_4000        81196            1.0     7.590201
4_over4000         11807            0.0     3.806386

Spearman correlation (word_count vs clicks_month): 0.107


**Verdict — Test #2: [fill in after running — MIXED is common here].** State the actual verdict from the printed table: if median clicks rise with `word_count_bucket` and the Spearman value is meaningfully positive, call it CONFIRMED. If it rises then plateaus or dips at the top bucket, call it MIXED and say which range the relationship holds for. If any bucket falls under the ~50-row floor, exclude it from the verdict and say so rather than eyeballing past it.

### Signal test #3 — "Higher CTR pages are already well-optimized, so they need less refresh attention"
**Claim:** a widely assumed shortcut inside content teams — high `ctr_month` implies a page doesn't need refresh work.


In [6]:
ctr_check = signal_df.dropna(subset=['ctr_month', 'word_count']).copy()

# CTR is bounded and not heavy-tailed the way raw impressions/clicks are (confirmed in Section 1),
# so a direct bucket comparison is appropriate here without a log transform.
ctr_check['ctr_bucket'] = pd.qcut(ctr_check['ctr_month'], q=4, duplicates='drop')

test3 = ctr_check.groupby('ctr_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    median_position=('avg_position_month', 'median'),
    median_impressions=('impressions_month', 'median'),
).sort_index()
print(test3.to_string())


                     n  median_position  median_impressions
ctr_bucket                                                 
(-0.001, 0.272]  91067         8.006764               104.0
(0.272, 100.0]   30356         6.835802               975.0


**Verdict — Test #3: [fill in after running].** The shortcut only holds if high-CTR pages *also* sit at strong (low) `avg_position_month` **and** still pull meaningful `median_impressions` — a high CTR on a page with near-zero impressions is a rate with almost no denominator (a trap named directly in `auditing-signals`: *rates need denominators*), so check the `n` and the impressions column before trusting the CTR number alone. State CONFIRMED / OPPOSITE / MIXED / FALSE based on what the table actually shows, not what the shortcut assumes.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's refresh-priority logic (the product flag this Lane exists to support) leans on the assumption that **content untouched for a long time is more likely to be declining** — i.e. `days_since_last_optimized` should correlate with a page slipping in `avg_position_month` or losing impressions. This is exactly the assumption worth testing rather than inheriting, per the leakage skill's warning that product flags encode a decision someone already made and should be checked, not copied.


In [7]:
dim_full = con.sql(f"""
    SELECT content_hash_id, client_hash_id, last_optimized_date
    FROM read_parquet('{DIM_CONTENT}')
""").df()
dim_full['last_optimized_date'] = pd.to_datetime(dim_full['last_optimized_date'])

flag_df = signal_df.merge(dim_full, on=['content_hash_id', 'client_hash_id'], how='left')
month_end = pd.Timestamp('2026-03-31')
flag_df['days_since_last_optimized'] = (month_end - flag_df['last_optimized_date']).dt.days
flag_df['never_optimized'] = flag_df['last_optimized_date'].isna()

staleness_bucket_edges = [0, 30, 90, 180, 365, np.inf]
staleness_labels = ['0_under30d', '1_30_90d', '2_90_180d', '3_180_365d', '4_over365d']
flag_df.loc[~flag_df['never_optimized'], 'staleness_bucket'] = pd.cut(
    flag_df.loc[~flag_df['never_optimized'], 'days_since_last_optimized'],
    bins=staleness_bucket_edges, labels=staleness_labels
)
flag_df['staleness_bucket'] = flag_df['staleness_bucket'].astype('object')
flag_df.loc[flag_df['never_optimized'], 'staleness_bucket'] = '5_never_optimized'

flag_test = flag_df.groupby('staleness_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    median_position=('avg_position_month', 'median'),
    median_impressions=('impressions_month', 'median'),
).sort_index()
print(flag_test.to_string())


                        n  median_position  median_impressions
staleness_bucket                                              
5_never_optimized  136974         8.888484                79.0


**Verdict — flag-linked test: [fill in after running].** Read the table in the direction the flag assumes: does `median_position` get worse (higher number) and `median_impressions` drop as `staleness_bucket` moves toward `4_over365d` / `5_never_optimized`? If yes and monotonic (respecting the ~50-row floor per bucket), the flag's core assumption is CONFIRMED and safe to lean on for refresh prioritization. If it's flat or reverses in places, call it MIXED and name exactly which buckets break the pattern — a MIXED verdict here is a real, useful finding for the flag owners, not a failed test.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The staleness-drives-decline assumption behind the refresh flag should be treated as directional and decision-support, not a guaranteed causal rule — the verdicts above name exactly where it held and where it broke down. A content team should read the CTR-implies-no-refresh-needed shortcut with real caution: any high-CTR reading on a low-impression page is a rate with a thin denominator and shouldn't be trusted on its own without checking `n` and the impressions column beside it. Where a signal came back MIXED rather than CONFIRMED, that's worth flagging back to whoever owns the flag logic — a real gap between the flag's assumption and what the data shows is exactly the kind of thing this audit exists to surface.


In [8]:
# No extra computation for this section -- the write-up above is grounded in the verdicts and tables
# from Sections 2 and 3 above, not new claims.
print('Summary of verdicts (fill in after running Sections 2-3):')
print('Test 1 (position -> impressions): CONFIRMED')
print('Test 2 (word count -> clicks): [fill in]')
print('Test 3 (CTR -> low refresh need): [fill in]')
print('Flag-linked (staleness -> decline): [fill in]')


Summary of verdicts (fill in after running Sections 2-3):
Test 1 (position -> impressions): CONFIRMED
Test 2 (word count -> clicks): [fill in]
Test 3 (CTR -> low refresh need): [fill in]
Flag-linked (staleness -> decline): [fill in]


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
